# Evaluation Comparison

Compares all fine-tuned models across:
- **Perplexity** (intrinsic language modeling quality)
- **ROUGE-1/2/L** (n-gram overlap with references)
- **BLEU** (precision-oriented translation metric)
- **Generation strategies** (greedy, beam search, top-k, top-p, temperature)
- **Resource usage** (time, CPU, memory per model)

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import pandas as pd
from pathlib import Path

from src.data.loader import load_stacksample
from src.data.preprocessing import preprocess_qa_pairs, split_dataset
from src.models.loader import load_tokenizer, load_model
from src.evaluation.evaluator import evaluate_model
from src.utils.config import load_config

In [ ]:
# ── Load test data ──────────────────────────────────────────────
questions, answers = load_stacksample(data_dir="../data")
qa_pairs = preprocess_qa_pairs(questions, answers)
_, test_df, _ = split_dataset(qa_pairs)

print(f"Test examples: {test_df.height:,}")

In [ ]:
MODEL_CONFIGS = [
    "../configs/pythia_410m.yaml",
    "../configs/gpt2_medium.yaml",
    "../configs/llama_3_1_8b.yaml",
    "../configs/qwen3_8b.yaml",
    "../configs/phi_3_5_moe.yaml",
    "../configs/phi_3_5_mini.yaml",
]

all_eval = {}

for config_path in MODEL_CONFIGS:
    cfg = load_config(config_path)
    output_dir = cfg["model"]["output_dir"]
    model_name = cfg["model"]["name"]

    if not Path(output_dir).exists():
        print(f"⚠ Skipping {model_name} — no fine-tuned weights at {output_dir}")
        continue

    print("\n" + "=" * 70)
    print(f"EVALUATING: {model_name}")
    print("=" * 70)

    device = "cuda" if torch.cuda.is_available() else "cpu"

    tokenizer = load_tokenizer(model_name)
    model = load_model(
        output_dir,
        device_map="auto",
        trust_remote_code=cfg["model"].get("trust_remote_code", False),
    )
    model.eval()

    results = evaluate_model(
        model, tokenizer, test_df, device,
        max_new_tokens=cfg["evaluation"]["max_new_tokens"],
        strategies=cfg["evaluation"]["strategies"],
    )

    all_eval[model_name] = results

    del model
    torch.cuda.empty_cache()

print("\n✓ Evaluation complete.")

In [ ]:
# ── Build comparison table ──────────────────────────────────────
rows = []
for model_name, res in all_eval.items():
    for strategy, metrics in res.items():
        if strategy == "perplexity":
            continue
        rows.append({
            "Model": model_name.split("/")[-1],
            "Strategy": strategy,
            "Perplexity": f"{res['perplexity']:.2f}",
            "ROUGE-1": f"{metrics['rouge1']:.4f}",
            "ROUGE-2": f"{metrics['rouge2']:.4f}",
            "ROUGE-L": f"{metrics['rougeL']:.4f}",
            "BLEU": f"{metrics['bleu']:.4f}",
            "Avg Time (s)": f"{metrics['avg_time']:.2f}",
        })

df = pd.DataFrame(rows)
print("\n" + "=" * 80)
print("EVALUATION COMPARISON — ALL MODELS × ALL STRATEGIES")
print("=" * 80)
display(df)

In [ ]:
# ── Best strategy per metric ────────────────────────────────────
for metric in ["rouge1", "rouge2", "rougeL", "bleu"]:
    print(f"\nBest {metric.upper()}:")
    for model_name, res in all_eval.items():
        best = max(
            ((s, m[metric]) for s, m in res.items() if s != "perplexity"),
            key=lambda x: x[1],
        )
        print(f"  {model_name.split('/')[-1]:35s} → {best[0]:25s} ({best[1]:.4f})")